# pipeline

> The headless decomposition pipeline (stage 5: decomp is an EXTENDER). Load a
> transcription run manifest, verify the transcription-emitted graph root exists
> (the graph begins at transcription), then per source per pipeline-segment run
> VAD + per-transcriber forced alignment, build one aligned segment per VAD chunk
> with per-transcriber text variants, and attach the fine spine under the existing
> AudioSegment nodes via the layer's idempotent `extend_graph` — with HITL approval
> seams between alignment, commit, and the next source.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_plugin_system.core.ports import (
    Composition, CompositionNode, CompositionRun, NodeState, OutputRef,
)
from cjm_plugin_system.core.empirical_store import compute_config_hash

# Typed wire-kind registration (stage 2): importing the DTO classes is what
# lets the proxy's wire_decode hand this host process TYPED results.
from cjm_media_plugin_system.core import MediaAnalysisResult
from cjm_forced_alignment_adapter_interface.core import ForcedAlignResult

# Stage 5: layer-owned plumbing + provenance-by-declaration.
from cjm_context_graph_layer.ops import graph_task, extend_graph
from cjm_context_graph_layer.declare import Derivation, derivation_to_graph

from cjm_transcript_decomp_core.models import (
    DecompConfig, DecompSegment, SegmentVariant, DecompSourceRecord, DecompManifest,
    FAWord, VADChunk, new_run_id,
)
from cjm_transcript_decomp_core.alignment import (
    map_fa_words_to_text, assign_words_to_chunks, build_segments_from_alignment,
    tier1_alignment_checks,
)
from cjm_transcript_decomp_core.graph import (
    resolve_root_ids, build_extension_payload, verify_source,
)

logger = logging.getLogger(__name__)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,   # Started job queue
    instance_id: str,  # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,          # Forwarded to the capability's execute()
) -> Any:  # The completed job's result payload
    """Submit one capability job, wait for it, and return its result (raise on failure)."""
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
def load_source_manifest(
    path: str,  # Path to a transcription-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    """Load + lightly validate a transcription-core run manifest.

    Consumed as untyped JSON — the format/version tags are the only interchange
    contract (no shared schema type across cores; manifest-as-interchange, CR-20).
    """
    data = json.loads(Path(path).read_text())
    fmt = data.get("format", "")
    if "transcription-core" not in fmt:
        logger.warning(f"unexpected source manifest format: {fmt!r} (continuing)")
    return data

In [ ]:
#| export
def vad_chunks_from_result(
    result: MediaAnalysisResult,  # Typed VAD result (wire-decoded at the proxy)
) -> List[VADChunk]:  # Segment-local VAD chunks, sorted, re-indexed
    """Normalize a typed VAD result into segment-local VAD chunks.

    Pure normalizer (stage 3): the capability invocation moved into the
    per-source composition (`build_alignment_composition`); this folds one
    node's typed result. Per-segment VAD (not whole-source) is the validated
    decomp path — originally a browser Web-Audio constraint, kept on the
    merits (D11).
    """
    chunks = [VADChunk(index=0, start_time=float(r.start), end_time=float(r.end))
              for r in result.ranges]
    chunks.sort(key=lambda c: c.start_time)
    for i, c in enumerate(chunks):
        c.index = i
    return chunks

In [ ]:
#| export
def fa_words_from_result(
    result: "ForcedAlignResult",  # Typed FA result (wire-decoded at the proxy)
) -> List[FAWord]:  # Word-level alignment results
    """Normalize a typed forced-alignment result into FA words (pure; stage 3)."""
    return [FAWord(text=str(it.text), start_time=float(it.start_time),
                   end_time=float(it.end_time))
            for it in result.items]


def build_alignment_composition(
    seg_list: List[Dict[str, Any]],  # Pipeline-segment entries from the transcription manifest (0.2.0)
    vad_id: str,               # VAD capability instance id
    fa_id: str,                # Forced-alignment capability instance id
    transcribers: List[str],   # Transcriber names to align (manifest `transcripts` keys, stable order)
    force: bool = False,       # Per-call cache-bypass control flag
) -> Tuple[Composition, List[Dict[str, Any]]]:  # (composition, per-pseg meta rows)
    """Build the whole-source M×(VAD ∥ T×FA) composition (D8 fan-in, stage-5 variants).

    Each pipeline segment contributes one VAD node + one FA node PER TRANSCRIBER
    with non-empty text — VAD is audio-only (run once; the shared skeleton), FA
    maps each transcriber's text onto it. All nodes are independent; the host's
    alignment fold fans them in. Pipeline segments where EVERY transcriber's
    text is empty are recorded as skipped meta rows and contribute no nodes.
    """
    nodes: List[CompositionNode] = []
    metas: List[Dict[str, Any]] = []
    for i, pseg in enumerate(seg_list):
        # Manifest entries are JSON dicts (manifest-as-interchange, CR-20).
        model_input = str(pseg.get("model_input_path", ""))
        seg_start = float(pseg.get("start", 0.0))
        transcripts = pseg.get("transcripts") or {}
        texts = {t: str((transcripts.get(t) or {}).get("text") or "")
                 for t in transcribers if t in transcripts}
        nonempty = {t: x for t, x in texts.items() if x.strip()}
        if not nonempty:
            metas.append({"skipped": True, "seg_start": seg_start, "pseg_index": i})
            continue
        vad_n = f"vad_{i:04d}"
        nodes.append(CompositionNode(vad_n, vad_id,
                                     {"media_path": model_input, "force": force}))
        fa_nodes: Dict[str, str] = {}
        for ti, t in enumerate(transcribers):
            if t not in nonempty:
                continue
            fa_n = f"fa_t{ti}_{i:04d}"
            nodes.append(CompositionNode(fa_n, fa_id,
                                         {"audio": model_input, "text": nonempty[t], "force": force}))
            fa_nodes[t] = fa_n
        metas.append({"skipped": False, "seg_start": seg_start, "pseg_index": i,
                      "vad_node": vad_n, "fa_nodes": fa_nodes, "texts": nonempty})
    return Composition(nodes=nodes), metas

In [ ]:
#| export
async def decompose_source(
    queue: JobQueue,
    cfg: DecompConfig,         # Run configuration
    source: Dict[str, Any],    # One source entry from the transcription manifest
    source_index: int,         # Position of this source within the run
    transcribers: List[str],   # Transcriber names (manifest order)
    text_from: str,            # Authoritative transcriber (layer-0 text)
) -> Tuple[str, List[DecompSegment]]:  # (source_path, ordered aligned segments)
    """Decompose one source into aligned fine segments with per-transcriber variants.

    The VAD-chunk skeleton is computed ONCE per pipeline segment (audio-only);
    each transcriber's text is forced-aligned onto it independently, then the
    pure fold merges per chunk: the `text_from` transcriber's alignment becomes
    the authoritative text, every transcriber's chunk text + char range rides
    `variants` (slice refs at commit). C4's "same skeleton, different text"
    duplication is gone — agreement is stored once by construction.
    """
    source_path = str(source.get("source_path", ""))
    seg_list = list(source.get("segments") or [])

    comp, metas = build_alignment_composition(
        seg_list, cfg.vad_plugin, cfg.fa_plugin, transcribers, force=cfg.force)
    results: Dict[str, Any] = {}
    if comp.nodes:
        comp_id = await queue.submit_composition(comp)
        crun = await queue.wait_for_composition(comp_id)
        if crun.status != NodeState.completed:
            failed = {nid: str(nr.error) for nid, nr in crun.node_runs.items()
                      if nr.state == NodeState.failed}
            raise RuntimeError(
                f"alignment composition {crun.status.value} for {source_path}: {failed}")
        results = crun.results_by_node()

    aligned: List[DecompSegment] = []
    global_index = 0
    for m in metas:
        seg_start = m["seg_start"]
        if m["skipped"]:
            logger.warning(f"[src {source_index}] no transcriber text at {seg_start:.1f}s; "
                           f"skipping pipeline segment")
            continue
        vad_chunks = vad_chunks_from_result(results[m["vad_node"]])

        # Per-transcriber fold over the SHARED skeleton.
        per_t_segments: Dict[str, List[Any]] = {}
        for t, fa_n in m["fa_nodes"].items():
            text = m["texts"][t]
            fa_words = fa_words_from_result(results[fa_n])
            spans = map_fa_words_to_text(text, fa_words)
            assignments = assign_words_to_chunks(fa_words, vad_chunks)
            per_t_segments[t] = build_segments_from_alignment(
                text=text, spans=spans, assignments=assignments,
                num_chunks=len(vad_chunks), source_provider_id=t,
            )
        auth = per_t_segments.get(text_from)
        if auth is not None:
            for w in tier1_alignment_checks(auth, vad_chunks):
                logger.warning(f"[src {source_index}] pseg @ {seg_start:.1f}s [{text_from}]: {w}")
        else:
            logger.warning(f"[src {source_index}] pseg @ {seg_start:.1f}s: authoritative "
                           f"transcriber {text_from!r} has no text here — layer-0 text empty")

        for k, vc in enumerate(vad_chunks):
            variants = []
            for t in transcribers:
                tsegs = per_t_segments.get(t)
                if tsegs is None:
                    continue
                ts = tsegs[k]
                if ts.text:
                    variants.append(SegmentVariant(transcriber=t, text=ts.text,
                                                   start_char=ts.start_char, end_char=ts.end_char))
            aligned.append(DecompSegment(
                index=global_index,
                text=(auth[k].text if auth is not None else ""),
                start_time=seg_start + vc.start_time, end_time=seg_start + vc.end_time,
                chunk_start=vc.start_time, chunk_end=vc.end_time,
                vad_chunk_index=vc.index, pseg_index=m["pseg_index"],
                variants=variants,
            ))
            global_index += 1
        logger.info(f"[src {source_index}] pseg @ {seg_start:.1f}s -> "
                    f"{len(vad_chunks)} chunks, {len(m['fa_nodes'])} transcriber alignment(s)")
    return source_path, aligned

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label, e.g. "alignment-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 warnings
      2. deterministic pre-filter: tier1_alignment_checks (no AI)
      3. modality-bridge candidate: spectrogram / word-overlay review (future Tier 2)
      4. authoritative verifier: re-align or re-transcribe-and-compare (future Tier 3)
      5. flywheel capture: accept/abort decisions logged; durable capture is a
         pass-2 seam-contract concern, not solved here

    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path, config_hash}
    """Record capability identity + data-DB pointers for the run manifest (provenance).

    Stage 5: also records each capability's EFFECTIVE config hash (the same
    `compute_config_hash` the empirical store keys on) — the VAD config hash is
    a Segment identity input. `db_path` prefers the effective config over the
    manifest default (a --graph-db-path override must be what downstream cores
    resolve; the D19 lesson). Stage 6 (0.2.1): the EFFECTIVE config dict is
    recorded READABLY beside its hash (the I8 lesson).
    """
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        current_config: Dict[str, Any] = {}
        try:
            proxy = manager.get_plugin(iid)
            if proxy is not None:
                current_config = proxy.get_current_config() or {}
        except Exception as e:  # Best-effort: identity recording must not fail the run
            logger.warning(f"collect_plugin_info: get_current_config({iid}) failed: {e}")
        info[iid] = {
            "name": meta.name,
            "version": getattr(meta, "version", None),
            "db_path": current_config.get("db_path") or manifest.get("db_path"),
            "config_hash": compute_config_hash(current_config),
            "config": current_config,
        }
    return info

In [ ]:
#| export
async def run_decomp(
    manager: PluginManager,        # Manager with VAD + FA + graph capabilities loaded
    queue: JobQueue,               # Started job queue
    cfg: DecompConfig,             # Run configuration
    source_manifest_path: str,     # Transcription run manifest to decompose
    run_id: Optional[str] = None,  # Override run id (default: generated)
) -> DecompManifest:  # Manifest of the sources extended
    """Extend every source in a transcription run manifest with its fine spine.

    Stage 5 (decomp-as-extender): the graph ROOT must already exist — decomp
    recomputes the deterministic root ids from the manifest, verifies the
    Source node is present (loud error pointing at transcription emission
    otherwise), aligns per transcriber, and attaches the fine spine via the
    layer's idempotent `extend_graph` (re-runs verify-collide). The fold is
    declared as a `Derivation` event when segments were actually created.
    """
    run_id = run_id or new_run_id()
    src = load_source_manifest(source_manifest_path)
    src_cfg = src.get("config", {}) or {}
    transcribers = list(src_cfg.get("transcriber_plugins") or [])
    if not transcribers and src_cfg.get("transcriber_plugin"):
        # pre-0.2.0 manifest: single transcriber under the old key
        transcribers = [str(src_cfg["transcriber_plugin"])]
    if not transcribers:
        raise RuntimeError("source manifest lists no transcribers")
    text_from = cfg.text_from or (transcribers[0] if len(transcribers) == 1 else None)
    if text_from is None:
        raise RuntimeError(
            f"multi-transcriber manifest ({transcribers}) requires --text-from "
            "(the authoritative transcriber for layer-0 text)")
    if text_from not in transcribers:
        raise RuntimeError(f"--text-from {text_from!r} not among the manifest's transcribers {transcribers}")
    src_plugins = src.get("plugins", {}) or {}

    manifest = DecompManifest(
        run_id=run_id, created_at=time.time(), config=cfg.to_dict(),
        source_manifest=str(source_manifest_path),
        source_format=src.get("format", ""), source_version=src.get("version", ""),
        plugins=collect_plugin_info(manager, [cfg.vad_plugin, cfg.fa_plugin, cfg.graph_plugin]),
    )
    vad_config_hash = str((manifest.plugins.get(cfg.vad_plugin) or {}).get("config_hash") or "")

    sources = src.get("sources", []) or []
    for i, source in enumerate(sources):
        # Extender pre-check: the transcription-emitted root must exist.
        roots = resolve_root_ids(source, src_plugins)
        root = await graph_task(queue, cfg.graph_plugin, "get_node", node_id=roots["source"])
        if root is None:
            raise RuntimeError(
                f"Source root {roots['source']} not found in the graph for "
                f"{source.get('source_path')!r} — the graph begins at transcription: "
                "re-run cjm-transcription-core with --graph-plugin against this DB first")

        source_path, aligned = await decompose_source(
            queue, cfg, source, i, transcribers, text_from)
        title = Path(source_path).stem or f"source-{i}"

        empty = sum(1 for a in aligned if not a.text.strip())
        warns = [f"{empty}/{len(aligned)} aligned segment(s) have empty layer-0 text"] if empty else []
        if not confirm_seam("alignment-review",
                            [f"{title}: {len(aligned)} aligned segment(s), text_from={text_from}"],
                            warns, assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: aborted at source {i} ({source_path})")
            break

        nodes, edges, ids = build_extension_payload(
            source, src_plugins, vad_config_hash, text_from, aligned)

        if not confirm_seam("commit-review",
                            [f"{title}: extending Source {ids['source'][:8]}… with "
                             f"{len(ids['segments'])} segment node(s) + {len(edges)} edge(s) "
                             f"via {cfg.graph_plugin}"],
                            [], assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: commit declined at source {i} ({source_path})")
            break

        res = await extend_graph(queue, cfg.graph_plugin, nodes, edges)
        logger.info(f"[src {i}] extension: +{res.nodes_added} node(s) "
                    f"({res.nodes_verified} verified present), +{res.edges_added} edge(s) "
                    f"({res.edges_existing} existing)")
        if res.nodes_added:
            d = Derivation(
                actor="host:cjm-transcript-decomp-core", method="alignment-fold/v1",
                input_ids=ids["transcripts_used"], output_ids=[ids["source"]],
                properties={"run_id": run_id, "segments": len(ids["segments"]),
                            "text_from": text_from},
            )
            dn, de = derivation_to_graph(d)
            await extend_graph(queue, cfg.graph_plugin, [dn], de)

        vr = await verify_source(queue, cfg.graph_plugin, ids["source"])
        if vr is None:
            logger.error(f"[src {i}] verify: Source {ids['source']} NOT FOUND in graph")
        else:
            logger.info(f"[src {i}] verify: {'OK' if vr.ok else 'FAILED'} "
                        f"(asegs={vr.audio_segment_count}, segments={vr.segment_count}, "
                        f"src_starts={vr.source_starts_with}, aseg_next={vr.aseg_next_complete}, "
                        f"seg_next={vr.seg_next_complete}, part_of={vr.part_of_complete}, "
                        f"aseg_starts={vr.aseg_starts_with_complete}, "
                        f"timing={vr.all_have_timing}, sources={vr.all_have_sources})")

        manifest.sources.append(DecompSourceRecord(
            source_node_id=ids["source"], source_path=source_path, title=title,
            segment_count=len(ids["segments"]), segment_ids=ids["segments"]))
    return manifest

In [ ]:
# pipeline import smoke check (no capabilities involved)
assert callable(run_decomp)
assert callable(decompose_source)
_m = load_source_manifest  # symbol present
print("pipeline import checks OK")

pipeline import checks OK


In [ ]:
# Composition builder + normalizer pure-logic checks (no capabilities involved)
from cjm_media_plugin_system.core import MediaAnalysisResult, TimeRange
from cjm_forced_alignment_adapter_interface.core import ForcedAlignResult, ForcedAlignItem
from cjm_plugin_system.core.ports import new_composition_run

# Normalizers fold typed wire results.
_chunks = vad_chunks_from_result(MediaAnalysisResult(
    ranges=[TimeRange(start=5.0, end=9.0), TimeRange(start=0.5, end=2.0)]))
assert [(c.index, c.start_time) for c in _chunks] == [(0, 0.5), (1, 5.0)]
_words = fa_words_from_result(ForcedAlignResult(
    items=[ForcedAlignItem(text="hi", start_time=0.1, end_time=0.4)]))
assert _words[0].text == "hi" and _words[0].end_time == 0.4

# Whole-source M×(VAD ∥ T×FA) shape (stage 5: per-transcriber FA off the
# shared skeleton): one VAD node per pseg; one FA node per transcriber with
# non-empty text; psegs where ALL transcribers are empty skip entirely.
_segs = [
    {"model_input_path": "/s0.wav", "start": 0.0,
     "transcripts": {"whisper": {"text": "alpha"}, "voxtral": {"text": "Alpha."}}},
    {"model_input_path": "/s1.wav", "start": 300.0,
     "transcripts": {"whisper": {"text": "  "}, "voxtral": {"text": ""}}},
    {"model_input_path": "/s2.wav", "start": 600.0,
     "transcripts": {"whisper": {"text": "beta"}, "voxtral": {"text": ""}}},
]
_comp, _metas = build_alignment_composition(_segs, "silero", "qwen3", ["whisper", "voxtral"])
# pseg0: vad + 2 FA; pseg1: skipped; pseg2: vad + 1 FA (voxtral empty there)
assert len(_comp.nodes) == 5 and len(_metas) == 3
assert _metas[1]["skipped"] is True and _metas[1]["pseg_index"] == 1
assert _metas[0]["fa_nodes"] == {"whisper": "fa_t0_0000", "voxtral": "fa_t1_0000"}
assert _metas[2]["fa_nodes"] == {"whisper": "fa_t0_0002"}
assert _comp.nodes[0].kwargs == {"media_path": "/s0.wav", "force": False}
assert _comp.nodes[2].kwargs == {"audio": "/s0.wav", "text": "Alpha.", "force": False}
_run = new_composition_run(_comp, "r")
assert set(_run.ready_nodes()) == {"vad_0000", "fa_t0_0000", "fa_t1_0000", "vad_0002", "fa_t0_0002"}
print("alignment composition builder/normalizer checks OK")